# Notebook 03: FlashAttention/SDPA Backend Comparison -- Real IO-Aware Latency & Memory

`[REAL]` Companion to Module 04. Uses `torch.nn.functional.scaled_dot_product_attention`'s real, selectable backends via `torch.nn.attention.sdpa_kernel` on the real RTX 4060.

**Scope of the claim (per signed-off plan):** this notebook measures a real **backend performance difference** (latency, peak memory) between SDPA kernels -- it does **not** claim to directly measure real HBM read/write traffic. Module 04's IO-complexity argument is the real, established *mechanism*; this notebook's numbers are consistent with that mechanism's real, expected consequences, but are not themselves an HBM-traffic measurement (that would require GPU-level profiling tools such as Nsight Compute, out of scope here).

In [1]:
import time
import statistics
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch version: {torch.__version__}")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
torch version: 2.13.0+cu126


## 1. Real Backend-Availability Check (Honest Fallback, Per Signed-Off Plan)

`[REAL]` Checking which real SDPA backends this exact PyTorch build actually supports on this GPU before running any timed comparison, using this model's real attention dimensions (14 heads, head_dim=64, matching `Qwen2.5-0.5B-Instruct`'s real query-head config from Notebook 02).

In [2]:
torch.manual_seed(0)
N_HEADS, D_HEAD = 14, 64

def make_qkv(batch_size, seq_len, n_heads=N_HEADS, d_head=D_HEAD):
    shape = (batch_size, n_heads, seq_len, d_head)
    q = torch.randn(shape, device=DEVICE, dtype=torch.float16)
    k = torch.randn(shape, device=DEVICE, dtype=torch.float16)
    v = torch.randn(shape, device=DEVICE, dtype=torch.float16)
    return q, k, v

test_q, test_k, test_v = make_qkv(1, 512)
backend_availability = {}
for name, backend in [("FLASH_ATTENTION", SDPBackend.FLASH_ATTENTION),
                      ("EFFICIENT_ATTENTION", SDPBackend.EFFICIENT_ATTENTION),
                      ("MATH", SDPBackend.MATH)]:
    try:
        with torch.no_grad(), sdpa_kernel(backend):
            _ = F.scaled_dot_product_attention(test_q, test_k, test_v, is_causal=True)
        backend_availability[name] = True
        print(f"{name}: available on this real hardware/build")
    except RuntimeError as e:
        backend_availability[name] = False
        print(f"{name}: NOT available on this real hardware/build -- {e}")

print("\n(pending real interpretation and backend selection for the timed comparison below)")

C:\Users\aryan\AppData\Local\Temp\ipykernel_22944\762547120.py:18: UserWarning: Memory efficient kernel not used because: (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:1104.)
  _ = F.scaled_dot_product_attention(test_q, test_k, test_v, is_causal=True)
C:\Users\aryan\AppData\Local\Temp\ipykernel_22944\762547120.py:18: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen/native/transformers/sdp_utils_cpp.h:571.)
  _ = F.scaled_dot_product_attention(test_q, test_k, test_v, is_causal=True)
C:\Users\aryan\AppData\Local\Temp\ipykernel_22944\762547120.py:18: UserWarning: Flash attention kernel not used because: (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:1106.)
  _ = F.scaled_dot_product_attention(test_q, test_k, test_v, is_causal=True)
C:\Users\aryan\AppDat

FLASH_ATTENTION: NOT available on this real hardware/build -- No available kernel. Aborting execution.
EFFICIENT_ATTENTION: available on this real hardware/build


MATH: available on this real hardware/build

(pending real interpretation and backend selection for the timed comparison below)


**Real, honest constraint found:** `FLASH_ATTENTION` is **not available** on this exact hardware/build — the real error is `"Torch was not compiled with flash attention"`, a genuine build-level limitation independent of this model's real GQA head-count mismatch (a separate real issue also observed when testing the actual model's own attention layers). Per the signed-off plan's fallback discipline, this constraint is reported honestly rather than worked around silently. `EFFICIENT_ATTENTION` (a genuine tiled/memory-efficient kernel in the same IO-aware family Module 04 discusses, conceptually) and `MATH` (the naive backend) **are** both available, so Section 2 compares those two real backends instead.

## 2. Real Latency & Peak Memory Comparison

`[REAL]` Comparing whichever two real backends Section 1 confirmed are available on this hardware/build, at a few real sequence lengths, using this model's real attention dimensions. Each measurement uses a real warm-up pass plus repeated timed runs with `torch.cuda.synchronize()`, matching Notebook 01's methodology.

In [3]:
def timed_sdpa(backend, q, k, v, n_repeats=8, n_warmup=1):
    def run():
        with torch.no_grad(), sdpa_kernel(backend):
            F.scaled_dot_product_attention(q, k, v, is_causal=True)

    for _ in range(n_warmup):
        run()
        if DEVICE == "cuda":
            torch.cuda.synchronize()

    samples = []
    for _ in range(n_repeats):
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        run()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        samples.append(time.perf_counter() - start)
    return statistics.median(samples), samples

def peak_memory_sdpa(backend, q, k, v):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), sdpa_kernel(backend):
        F.scaled_dot_product_attention(q, k, v, is_causal=True)
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated()

SEQ_LENGTHS = [512, 1024, 2048, 4096]
available_backends = [(name, b) for name, b in
                      [("FLASH_ATTENTION", SDPBackend.FLASH_ATTENTION),
                       ("EFFICIENT_ATTENTION", SDPBackend.EFFICIENT_ATTENTION),
                       ("MATH", SDPBackend.MATH)]
                      if backend_availability.get(name)]
print(f"Comparing real backends: {[n for n, _ in available_backends]}")

comparison_results = []
for L in SEQ_LENGTHS:
    q, k, v = make_qkv(1, L)
    row = {"seq_len": L}
    for name, backend in available_backends:
        median_s, _ = timed_sdpa(backend, q, k, v)
        peak_bytes = peak_memory_sdpa(backend, q, k, v)
        row[f"{name}_latency_ms"] = median_s * 1000
        row[f"{name}_peak_mb"] = peak_bytes / 1024 / 1024
    comparison_results.append(row)
    parts = [f"{name}: {row[f'{name}_latency_ms']:.3f}ms / {row[f'{name}_peak_mb']:.2f}MB" for name, _ in available_backends]
    print(f"L={L:5d}: " + ", ".join(parts))

print("\n(pending real interpretation)")

Comparing real backends: ['EFFICIENT_ATTENTION', 'MATH']
L=  512: EFFICIENT_ATTENTION: 0.193ms / 14.25MB, MATH: 1.031ms / 52.88MB
L= 1024: EFFICIENT_ATTENTION: 0.201ms / 17.75MB, MATH: 3.703ms / 160.01MB


L= 2048: EFFICIENT_ATTENTION: 0.704ms / 24.75MB, MATH: 14.503ms / 569.28MB


L= 4096: EFFICIENT_ATTENTION: 1.621ms / 38.75MB, MATH: 53.599ms / 2167.81MB

(pending real interpretation)


## 3. Real Interpretation

`[REAL]` `EFFICIENT_ATTENTION` real-measured faster and leaner than `MATH` at every real sequence length, with **both gaps growing as sequence length grows**:

| Seq len | Latency ratio (MATH / EFFICIENT) | Peak memory ratio (MATH / EFFICIENT) |
|---|---|---|
| 512 | `5.34x` | `3.71x` |
| 1024 | `18.42x` | `9.01x` |
| 2048 | `20.60x` | `23.00x` |
| 4096 | `33.07x` | `55.94x` |

**Real memory-scaling pattern, consistent with (not a direct measurement of) Module 04's IO-aware mechanism:** doubling sequence length scaled `MATH`'s real peak memory by `3.03x`, `3.56x`, then `3.81x` — approaching the quadratic ($O(L^2)$) growth expected from materializing the full attention score matrix. `EFFICIENT_ATTENTION`'s real peak memory scaled by only `1.25x`, `1.39x`, then `1.57x` per doubling — much closer to linear ($O(L)$) growth, consistent with never materializing that full matrix. **This is a real, measured latency/memory outcome consistent with the naive-vs-tiled IO-aware distinction Module 04 describes — it is not itself a measurement of real HBM read/write traffic**, per the scope stated in this notebook's own opening cell; a genuine HBM-traffic measurement would require GPU-level profiling tools (e.g., Nsight Compute) outside this notebook's scope.